# Phase 5 — RL Pit Agent (PPO) — Kaggle
Trains the PPO pit-strategy agent inside the race simulator (`src/rl/env.py`),
across **3 seeds**, then runs the head-to-head versus the Phase 4a MDP policy in the
SAME simulator (`src/rl/evaluate_rl.py`). A documented "RL matched but did not beat the
MDP" is a legitimate result (MASTER_CONTEXT 10.5) — this notebook reports it either way.

## Honest note on the hardware (READ THIS)
**PPO with an MLP policy is mostly CPU-bound.** The forward/backward passes on a tiny
MLP are dwarfed by the Python env-stepping, which runs on CPU. The 2x T4 GPUs do **not**
meaningfully speed up this job. The sensible use of the hardware here is:
- **`SubprocVecEnv` with `--n-envs 8`** — parallel env rollouts across CPU cores (the real
  throughput lever). Kaggle gives ~4 vCPUs, so 8 envs oversubscribe slightly, which is fine.
- **Run the 3 seeds sequentially** in one session (default below), OR launch two seeds as
  separate processes pinned one per GPU (`--device cuda`) if and only if a GPU run profiles
  faster on your config — usually it does not for an MLP. Keep `--device cpu` unless measured.

## Budget (~10 h total, per the playbook)
3 seeds x 2-5M steps, checkpoints every 500k. At ~300-400 env-steps/s on CPU with 8 envs,
~3M steps is ~2.5-3.5 h per seed -> ~8-10 h for 3 seeds. Tune `STEPS` to your session budget
(Kaggle caps a session at **12 h**). Checkpoints every 500k mean a dead session loses < 30 min.

## Before running
1. Notebook settings: **Accelerator = GPU T4 x2** (for parity with other runs; PPO uses CPU
   unless you pass `--device cuda`), **Internet = ON**, **Persistence = Files only** (so
   checkpoints survive a restart).
2. **+ Add Input** -> attach the `tft_full_data` dataset (the `laps_*_r*.parquet` files,
   same dataset as notebooks 04-07). The env needs `models/tyre_curves.joblib`, which we
   regenerate from these parquets in cell 4 (fast, CPU).
3. Repo pushed to GitHub `main` (cell 2 clones/pulls it). `data/pit_loss.json`,
   `models/tft_calibration.json`, and `reports/lap_time/tft_recalibration.csv` are committed,
   so they arrive with the clone.

**When done:** download `rl_ppo_artifacts.zip` from the Output panel (models + tb logs +
eval csv). Local steps in the last cell.

In [ ]:
# 1. Deps: stable-baselines3 + gymnasium (Section 3 RL block). Keep Kaggle's stock torch.
!pip -q install stable-baselines3 gymnasium tensorboard fastf1 pandera scipy joblib
import stable_baselines3 as sb3, gymnasium as gym, torch
print('sb3', sb3.__version__, '| gymnasium', gym.__version__, '| torch', torch.__version__)
print('cuda:', torch.cuda.is_available())

In [ ]:
# 2. Clone repo (or pull) + path
import os, sys
REPO = '/kaggle/working/f1-strategist'
if not os.path.exists(REPO):
    !git clone https://github.com/Shreyansh262/f1-strategist.git $REPO
else:
    !cd $REPO && git pull
sys.path.insert(0, REPO)
os.chdir(REPO)
!cd $REPO && git log --oneline -1

In [ ]:
# 3. Copy parquets from the attached dataset into data/raw (same as runs 04-07)
import glob, shutil, pathlib
dst = pathlib.Path(REPO) / 'data' / 'raw'
dst.mkdir(parents=True, exist_ok=True)
src_files = glob.glob('/kaggle/input/**/laps_*_r*.parquet', recursive=True)
assert src_files, 'No laps_*_r*.parquet under /kaggle/input/ — attach the data dataset first.'
for f in src_files:
    shutil.copy(f, dst / pathlib.Path(f).name)
copied = sorted(p.name for p in dst.glob('laps_*_r*.parquet'))
print(len(copied), 'files | seasons:', sorted({n.split('_')[1] for n in copied}))

In [ ]:
# 4. Ensure the env's model inputs exist.
#    The RaceEnv needs models/tyre_curves.joblib (gitignored). Regenerate it from the
#    parquets via the Phase 3 fit (fast, CPU). pit_loss.json / tft_calibration.json /
#    tft_recalibration.csv are committed and arrived with the clone.
import pathlib
curves = pathlib.Path(REPO) / 'models' / 'tyre_curves.joblib'
if not curves.exists():
    print('Fitting tyre curves (one-off, ~1-2 min)...')
    from src.models.tyre.fit import main as fit_tyres
    fit_tyres()
assert curves.exists(), 'tyre_curves.joblib still missing'
# sanity: build one env
from src.rl.env import RaceEnv
from gymnasium.utils.env_checker import check_env
check_env(RaceEnv(), skip_render_check=True)
print('env OK — check_env passed')

## Train — 3 seeds (sequential, parallel envs)
Each seed: PPO MlpPolicy, `n_envs=8` (SubprocVecEnv), checkpoints every 500k to
`models/rl/`. Set `STEPS` to fit your session. The trainer is resumable in spirit —
if a session dies, completed seeds' final `.zip` files persist (Persistence = Files only);
re-run this cell and skip seeds whose final artifact already exists.

In [ ]:
# 5. Train the 3 seeds
from pathlib import Path
from src.rl.train_ppo import train, MODELS_DIR

STEPS  = 3_000_000     # 2-5M per the playbook; 3M ~ a sensible mid-point
N_ENVS = 8             # parallel env rollouts (the real throughput lever on CPU)
SEEDS  = [0, 1, 2]
DEVICE = 'cpu'         # MLP PPO is CPU-bound; only flip to 'cuda' if you profile a win

for seed in SEEDS:
    final = MODELS_DIR / f'ppo_pit_seed{seed}.zip'
    if final.exists():
        print(f'seed {seed}: final artifact exists, skipping'); continue
    print(f'=== training seed {seed} ===', flush=True)
    train(steps=STEPS, seed=seed, n_envs=N_ENVS, device=DEVICE, checkpoint_freq=500_000)
print('All seeds done. Artifacts in', MODELS_DIR)

## Evaluate — PPO vs MDP, paired (common random numbers)
Both policies roll out the same RaceEnv episodes under identical seeds, so the
difference is the policy, not luck. We evaluate each seed's final policy; the
best-by-mean-finish seed is the headline.

In [ ]:
# 6. Head-to-head for each seed
import pandas as pd
from src.rl.evaluate_rl import evaluate, REPORTS_DIR

EPISODES = 500
summary = []
for seed in SEEDS:
    model = MODELS_DIR / f'ppo_pit_seed{seed}.zip'
    if not model.exists():
        print(f'seed {seed}: no model, skipping'); continue
    out = evaluate(model, episodes=EPISODES)
    out.to_csv(REPORTS_DIR / f'ppo_vs_mdp_seed{seed}.csv', index=False)
    agg = out[out['seed'] == 'AGGREGATE'].iloc[0]
    summary.append({'seed': seed, 'ppo_mean_finish': agg['ppo_finish'],
                    'mdp_mean_finish': agg['mdp_finish'],
                    'ppo_minus_mdp_time_s': agg['ppo_minus_mdp_time_s'],
                    'ppo_legal_frac': agg['ppo_legal']})
summary = pd.DataFrame(summary)
summary.to_csv(REPORTS_DIR / 'ppo_vs_mdp.csv', index=False)
print(summary.to_string(index=False))
print('\nReading: negative ppo_minus_mdp_time_s = PPO faster than the MDP (a beat).')

In [ ]:
# 7. Bundle artifacts for download (Output panel): models + tb logs + eval csvs.
from pathlib import Path
(Path(REPO) / 'models' / 'rl').mkdir(parents=True, exist_ok=True)
(Path(REPO) / 'reports' / 'rl').mkdir(parents=True, exist_ok=True)
!cd $REPO && zip -qr /kaggle/working/rl_ppo_artifacts.zip models/rl reports/rl
print('Download: /kaggle/working/rl_ppo_artifacts.zip')
!ls -la $REPO/models/rl $REPO/reports/rl

## After downloading (LOCAL steps)
1. Unzip `rl_ppo_artifacts.zip` into the repo root (populates `models/rl/*.zip` and
   `reports/rl/*.csv`; model binaries stay gitignored, the report CSVs get committed).
2. Sanity-check locally: `python -m src.rl.evaluate_rl --model models/rl/ppo_pit_seed0.zip --episodes 200`.
3. `pytest -q` — all green (RL env tests don't need the trained model).
4. Write up the result honestly in the model card + MASTER_CONTEXT Section 8 —
   whether PPO beat, matched, or lost to the MDP, with the paired-comparison numbers.